In [ ]:
import re
import time
import random
import unicodedata
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "http://www.guiadosquadrinhos.com"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; AcademicCrawler/1.0; +contact: virginio.ruan@academico.ifpb.edu.br)"
}

MESES_PT = {
    "janeiro": 1, "fevereiro": 2, "março": 3, "abril": 4, "maio": 5, "junho": 6,
    "julho": 7, "agosto": 8, "setembro": 9, "outubro": 10, "novembro": 11, "dezembro": 12
}

NEXT_TEXTS = {">", "›", "»", ">>", "próxima", "proxima", "próximo", "proximo"}

# /edicao/<slug>/<codigo>/<id>
EDICAO_LINK_RE = re.compile(r"/edicao/[^\"'\s]+/[^\"'\s]+/\d+$", re.I)

def sleep_polite(a=2.5, b=6.0): # um sleep generoso pra não prejudicar os servidores do guiadosquadrinhos
    time.sleep(random.uniform(a, b))

def get_soup(html):
    return BeautifulSoup(html, "html.parser")

def fetch(url, session):
    r = session.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def get_text_after_label(soup, label):
    strong = soup.find("strong", string=re.compile(rf"^{re.escape(label)}\s*$", re.I))
    if not strong:
        return None
    node = strong.next_sibling
    while node and isinstance(node, str) and node.strip() == "":
        node = node.next_sibling
    if not node:
        return None
    return node.get_text(" ", strip=True) if hasattr(node, "get_text") else str(node).strip()

def parse_mes_ano(texto):
    if not texto:
        return (None, None)
    m = re.search(r"([A-Za-zçãõéíóúâêôà]+)\s+de\s+(\d{4})", texto.strip(), re.I)
    if not m:
        return (None, None)
    mes = MESES_PT.get(m.group(1).lower())
    ano = int(m.group(2))
    return (mes, ano)

def parse_issue_number(soup):
    h1 = soup.find("h1")
    if not h1:
        return None
    txt = h1.get_text(" ", strip=True)
    m = re.search(r"n[º°]\s*(\d+)", txt, flags=re.I)
    return int(m.group(1)) if m else None

def find_next_edicao_url(soup, current_url):
    # pega número atual (pra usar no fallback)
    cur_num = parse_issue_number(soup)

    # rel="next" ou title indicando próxima
    for a in soup.find_all("a", href=True):
        title = (a.get("title") or "").lower()
        rel = " ".join(a.get("rel", [])).lower() if a.get("rel") else ""
        if ("next" in rel) or ("próxima" in title) or ("proxima" in title):
            full = urljoin(current_url, a["href"].strip())
            if EDICAO_LINK_RE.search(full):
                return full

    # texto do link (>, Próxima, etc.)
    for a in soup.find_all("a", href=True):
        txt = a.get_text(" ", strip=True).lower()
        if txt in NEXT_TEXTS or any(t in txt for t in ["próxima", "proxima", "próximo", "proximo"]):
            full = urljoin(current_url, a["href"].strip())
            if EDICAO_LINK_RE.search(full):
                return full

    # fallback por lista de links de edição: escolhe a "próxima" pelo número no slug
    # exemplo de slug: /edicao/batman-2-serie-n-1/ba001200/30304
    candidates = []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        full = urljoin(current_url, href)
        if not EDICAO_LINK_RE.search(full):
            continue

        # tenta extrair o número do "-n-XX" no slug
        m = re.search(r"/edicao/[^/]*-n-(\d+)/", full, flags=re.I)
        if m:
            n = int(m.group(1))
            candidates.append((n, full))

    if cur_num is not None and candidates:
        # pega o menor n > cur_num
        bigger = [c for c in candidates if c[0] > cur_num]
        if bigger:
            bigger.sort(key=lambda x: x[0])
            return bigger[0][1]

    return None

def parse_issue_page(html, url):
    soup = get_soup(html)
    numero = parse_issue_number(soup)

    publicado = get_text_after_label(soup, "Publicado em:")
    mes, ano = parse_mes_ano(publicado)

    preco = get_text_after_label(soup, "Preço de capa:")
    paginas = get_text_after_label(soup, "Número de páginas:")

    # Tenta converter páginas para inteiro
    try:
        paginas = int(re.search(r"\d+", paginas).group()) if paginas else None
    except:
        paginas = None

    return {
        "edicao_numero": numero,
        "mes": mes,
        "ano": ano,
        "preco_capa": preco,
        "numero_paginas": paginas,
        "url": url
    }, soup


def slugify(s: str) -> str:
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s or "sem_nome"

def crawl_follow_next(
    start_url: str,
    out_csv: str,
    max_edicoes: int = 2000,
    checkpoint_every: int = 50,
    polite_sleep: bool = True
):
    session = requests.Session()
    url = start_url

    rows = []
    seen_urls = set()

    for i in range(max_edicoes):
        if url in seen_urls:
            print("[stop] URL repetida, encerrando.")
            break
        seen_urls.add(url)

        html = fetch(url, session)
        data, soup = parse_issue_page(html, url)

        if data["edicao_numero"] is not None:
            rows.append(data)
            print(f"✓ edição {data['edicao_numero']}")

        if checkpoint_every and len(rows) % checkpoint_every == 0:
            (pd.DataFrame(rows)
             .drop_duplicates("edicao_numero")
             .sort_values("edicao_numero")
             .to_csv(out_csv, index=False, encoding="utf-8-sig"))
            print(f"[checkpoint] salvei {out_csv}")

        next_url = find_next_edicao_url(soup, url)
        if not next_url:
            print("Não achei link de próxima edição. Parando.")
            break

        url = next_url
        if polite_sleep:
            sleep_polite()

    df = pd.DataFrame(rows).drop_duplicates("edicao_numero").sort_values("edicao_numero")
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[fim] Salvei {len(df)} edições em {out_csv}")
    return df

# o batman já passou por tres editoras no brasil e com multiplas series ao longo dos anos
SERIES_BATMAN = [
    {
        "serie_nome": "batman_1_serie",
        "editora": "ebal",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-1-serie-n-1/ba001102/30204"
    },
    {
        "serie_nome": "batman_2_serie",
        "editora": "ebal",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-2-serie-n-1/ba001200/30304"
    },
    {
        "serie_nome": "batman_3_serie",
        "editora": "ebal",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-3-serie-n-1/ba001300/30404"
    },
        {
        "serie_nome": "batman_4_serie",
        "editora": "ebal",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-4-serie-n-1/ba001400/30493"
    },
        {
        "serie_nome": "batman_1_serie",
        "editora": "abril",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-1-serie-n-1/bt00301/4741"
    },
        {
        "serie_nome": "batman_2_serie",
        "editora": "abril",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-2-serie-n-1/bt00302/4751"
    },
        {
        "serie_nome": "batman_3_serie",
        "editora": "abril",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-3-serie-n-0/bt00303/20984"
    },
        {
        "serie_nome": "batman_4_serie",
        "editora": "abril",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-4-serie-n-1/bt00304/4796"
    },
        {
        "serie_nome": "batman_5_serie",
        "editora": "abril",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-5-serie-n-0/bt00305/4815"
    },
        {
        "serie_nome": "batman_6_serie",
        "editora": "abril",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-6-serie-n-1/bt00306/5022"
    },
        {
        "serie_nome": "batman_7_serie",
        "editora": "abril",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-7-serie-n-1/btn0301/4861"
    },
    {
        "serie_nome": "batman_1_serie",
        "editora": "panini",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-1-serie-n-1/ba01101/19506"
    },
        {
        "serie_nome": "batman_2_serie",
        "editora": "panini",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-2-serie-n-0/ba011203/104688"
    },
        {
        "serie_nome": "batman_3_serie",
        "editora": "panini",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-3-serie-n-1/ba011156/129574"
    },
        {
        "serie_nome": "batman_4_serie",
        "editora": "panini",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-4-serie-n-1/ba011400/163183"
    },
        {
        "serie_nome": "batman_5_serie",
        "editora": "panini",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-5-serie-n-1/ba011500/171848"
    },
        {
        "serie_nome": "batman_6_serie",
        "editora": "panini",
        "start_url": "http://www.guiadosquadrinhos.com/edicao/batman-6-serie-n-1/ba011600/186510"
    },
]

def crawl_multiseries(series_list, pasta_saida=".", max_edicoes=2000):
    resultados = {}
    for item in series_list:
        serie_nome = item["serie_nome"]
        editora = item["editora"]
        start_url = item["start_url"]

        out_csv = f"{pasta_saida}/Batman/edicoes_{slugify(serie_nome)}_{slugify(editora)}.csv"
        print("\n" + "="*70)
        print(f"Iniciando: {serie_nome} | {editora}")
        print(f"Start: {start_url}")
        print(f"Saída: {out_csv}")
        print("="*70)

        df = crawl_follow_next(
            start_url=start_url,
            out_csv=out_csv,
            max_edicoes=max_edicoes,
            checkpoint_every=50,
            polite_sleep=True
        )
        resultados[(serie_nome, editora)] = df

    return resultados

resultados = crawl_multiseries(SERIES_BATMAN, pasta_saida=".", max_edicoes=3000)